In [ ]:
from pathlib import Path
import gc
import importlib.util
import os
import subprocess
import sys
import time

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive as colab_drive
    colab_drive.mount("/content/drive", force_remount=False)
    WORKSPACE = Path("/content/drive/MyDrive/Zhong et al. 2025 - Neuromatch Team Workspace")
    CODE = WORKSPACE / "code"
    CACHE = Path("/content/zhong-cache")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy>=1.11,<2"], check=True)
    DATABASE = WORKSPACE / "zhong.duckdb"
else:
    WORKSPACE = next(
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "code").is_dir() and (path / "data" / "cache" / "zhong.duckdb").is_file()
    )
    CODE = WORKSPACE / "code"
    CACHE = WORKSPACE / "data" / "cache"
    DATABASE = CACHE / "zhong.duckdb"
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE / ".matplotlib"))
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import drive
from joiner import Joiner

db = drive.setup(cache=str(CACHE), database=str(DATABASE), mount=False)
assert db.database_path.is_file()
db


In [ ]:
import duckdb

from dprime import AREAS
from position import component_trial_position
from single_neuron import single_neuron_tuning_session

POSITION_EDGES = np.linspace(0.0, 4.0, 21)
LICK_LAGS = np.arange(-10, 11, dtype=np.int64)
OUT = drive.results("janviere")
OUT.mkdir(parents=True, exist_ok=True)
RESULT_DATABASE = OUT / "single_neuron_metrics.duckdb"
{"areas": AREAS, "position_bins": 20, "lick_lags": LICK_LAGS.tolist()}


In [ ]:
manifest = db.query("""
    SELECT b.behavior_session_id, b.behavior_key, b.recording_id,
           b.experiment, b.mouse, b.cohort, e.stage, e.moment, b.trial_count
    FROM behavior_sessions AS b
    JOIN recordings AS r USING (recording_id)
    JOIN experiments AS e USING (experiment)
    WHERE r.has_behavior AND r.has_reduced_neural AND r.has_retinotopy
    ORDER BY b.cohort, b.mouse, e.stage, e.moment, b.recording_id, b.behavior_key
""")
manifest


In [ ]:
assert len(manifest) == 142 and manifest["recording_id"].nunique() == 89
db.register("session_manifest", manifest)
db.query("""
    SELECT cohort, stage, moment,
           COUNT(DISTINCT behavior_session_id) AS sessions,
           COUNT(DISTINCT recording_id) AS recordings,
           COUNT(DISTINCT mouse) AS mice
    FROM session_manifest
    GROUP BY ALL
    ORDER BY cohort, stage, moment
""")


In [ ]:
probe = next(manifest.itertuples(index=False))
probe_joiner = Joiner(
    db,
    probe.recording_id,
    experiment=probe.experiment,
    behavior_key=probe.behavior_key,
)
probe_joiner.query("""
    SELECT area_group, area_id, COUNT(*) AS neurons
    FROM neurons
    GROUP BY ALL
    ORDER BY area_group, area_id
""")


In [ ]:
probe_frames, probe_response, probe_trials, _ = component_trial_position(
    probe_joiner, POSITION_EDGES
)
pd.DataFrame(
    {
        "frames": [len(probe_frames)],
        "trials": [len(probe_trials)],
        "components": [probe_response.shape[0]],
        "position_bins": [probe_response.shape[2]],
        "populated_bins": [np.isfinite(probe_response).any(axis=(0, 1)).sum()],
    }
)


In [ ]:
del probe_joiner, probe_response
gc.collect()

probe_metrics, probe_tuning, probe_lick = single_neuron_tuning_session(
    db, probe, POSITION_EDGES, LICK_LAGS, verify=True
)
probe_metrics[
    [
        "neuron_id", "area_group", "mean_activity", "run_correlation",
        "spatial_strength", "spatial_reliability", "preferred_position_m",
        "lick_modulation_z",
    ]
].head(12)


In [ ]:
db.register("probe_metrics_view", probe_metrics)
db.query("""
    SELECT area_group, COUNT(*) AS neurons,
           MEDIAN(run_correlation) AS run_correlation,
           MEDIAN(spatial_strength) AS spatial_strength,
           MEDIAN(spatial_reliability) AS spatial_reliability,
           MEDIAN(lick_modulation_z) AS lick_modulation
    FROM probe_metrics_view
    GROUP BY area_group
    ORDER BY area_group
""")


In [ ]:
result_db = duckdb.connect(str(RESULT_DATABASE))
result_db.execute("DROP TABLE IF EXISTS neuron_metrics")
tuning_scan = [probe_tuning]
lick_scan = [probe_lick] if not probe_lick.empty else []

result_db.register("batch", probe_metrics)
result_db.execute("CREATE TABLE neuron_metrics AS SELECT * FROM batch")
result_db.unregister("batch")

for index, session in enumerate(manifest.itertuples(index=False), start=1):
    if session.behavior_session_id == probe.behavior_session_id:
        continue
    started = time.perf_counter()
    metrics, tuning, lick = single_neuron_tuning_session(db, session, POSITION_EDGES, LICK_LAGS)
    result_db.register("batch", metrics)
    result_db.execute("INSERT INTO neuron_metrics SELECT * FROM batch")
    result_db.unregister("batch")
    tuning_scan.append(tuning)
    if not lick.empty:
        lick_scan.append(lick)
    print(f"{index:03d}/142 {session.behavior_session_id} {len(metrics):,} neurons {time.perf_counter() - started:.1f}s")

result_db.execute("""
    SELECT COUNT(*) AS neuron_session_rows,
           COUNT(DISTINCT behavior_session_id) AS sessions,
           COUNT(DISTINCT recording_id) AS recordings,
           COUNT(DISTINCT mouse) AS mice
    FROM neuron_metrics
""").fetchdf()


In [ ]:
result_db.execute("""
    SELECT area_group, COUNT(*) AS neuron_session_rows,
           MEDIAN(run_correlation) AS run_correlation,
           MEDIAN(spatial_strength) AS spatial_strength,
           MEDIAN(spatial_reliability) AS spatial_reliability,
           MEDIAN(lick_modulation_z) AS lick_modulation_z
    FROM neuron_metrics
    GROUP BY area_group
    ORDER BY area_group
""").fetchdf()


In [ ]:
exemplar_tuning = pd.concat(tuning_scan, ignore_index=True)
exemplar_lick = pd.concat(lick_scan, ignore_index=True) if lick_scan else pd.DataFrame()
exemplar_tuning.to_csv(OUT / "exemplar_spatial_tuning.csv", index=False)
exemplar_lick.to_csv(OUT / "exemplar_lick_triggered.csv", index=False)
{"tuning_rows": exemplar_tuning.shape, "lick_rows": exemplar_lick.shape}


In [ ]:
area_metrics = result_db.execute("""
    SELECT behavior_session_id, recording_id, experiment, mouse, cohort, stage, moment,
           area_group, COUNT(*) AS neurons,
           MEDIAN(mean_activity) AS median_activity,
           MEDIAN(activity_sd) AS median_activity_sd,
           MEDIAN(run_correlation) AS median_run_correlation,
           MEDIAN(spatial_strength) AS median_spatial_strength,
           MEDIAN(spatial_reliability) AS median_spatial_reliability,
           MEDIAN(lick_modulation_z) AS median_lick_modulation_z,
           AVG(CASE WHEN spatial_reliability >= 0.5 THEN 1.0 ELSE 0.0 END) AS reliable_fraction,
           MAX(lick_frames) AS lick_frames
    FROM neuron_metrics
    GROUP BY ALL
    ORDER BY cohort, mouse, stage, moment, behavior_session_id, area_group
""").fetchdf()
area_metrics.to_csv(OUT / "area_metrics.csv", index=False)
area_metrics.head(12)


In [ ]:
db.register("area_metrics_view", area_metrics)
mouse_metrics = db.query("""
    SELECT mouse, cohort, stage, moment, area_group,
           COUNT(DISTINCT behavior_session_id) AS sessions,
           AVG(median_activity) AS median_activity,
           AVG(median_run_correlation) AS median_run_correlation,
           AVG(median_spatial_strength) AS median_spatial_strength,
           AVG(median_spatial_reliability) AS median_spatial_reliability,
           AVG(median_lick_modulation_z) AS median_lick_modulation_z,
           AVG(reliable_fraction) AS reliable_fraction
    FROM area_metrics_view
    GROUP BY ALL
    ORDER BY cohort, mouse, stage, moment, area_group
""")
mouse_metrics.to_csv(OUT / "mouse_first_area_metrics.csv", index=False)
mouse_metrics.head(12)


In [ ]:
plot_data = db.query("""
    WITH mouse_area AS (
        SELECT mouse, cohort, area_group,
               AVG(median_run_correlation) AS run_correlation,
               AVG(median_spatial_reliability) AS spatial_reliability,
               AVG(median_lick_modulation_z) AS lick_modulation
        FROM area_metrics_view
        GROUP BY ALL
    )
    SELECT cohort, area_group, COUNT(*) AS mice,
           AVG(run_correlation) AS run_correlation,
           AVG(spatial_reliability) AS spatial_reliability,
           AVG(lick_modulation) AS lick_modulation
    FROM mouse_area
    GROUP BY ALL
    ORDER BY cohort, area_group
""")
plot_data


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for axis, metric in zip(axes, ["run_correlation", "spatial_reliability", "lick_modulation"]):
    plot_data.pivot(index="area_group", columns="cohort", values=metric).reindex(AREAS).plot.bar(
        ax=axis, legend=False
    )
    axis.set(title=metric.replace("_", " "), xlabel="")
    axis.axhline(0, color="0.5", linewidth=0.8)
axes[-1].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "single_neuron_area_comparison.png", dpi=180)
plt.show()


In [ ]:
result_db.close()
pd.DataFrame(
    {
        "file": ["single_neuron_metrics.duckdb", "area_metrics.csv", "mouse_first_area_metrics.csv"],
        "exists": [
            RESULT_DATABASE.is_file(),
            (OUT / "area_metrics.csv").is_file(),
            (OUT / "mouse_first_area_metrics.csv").is_file(),
        ],
    }
)
